In [12]:
from labjack import ljm
import time

# Open T7 connection
handle = ljm.openS("T7", "ANY", "ANY")

GAIN = 51.0
OFFSET_V = 1.25  # Built-in 1.25V offset of the LJTick-InAmp 3

def type_k_temp(tc_voltage_v, cj_temp_c):
    """
    Approximate Type K thermocouple conversion.
    tc_voltage_v is the ACTUAL thermocouple voltage extracted from the InAmp transfer function.
    """
    sensitivity = 41.276e-6  # V/°C near room temperature
    return cj_temp_c + tc_voltage_v / sensitivity

# Configure channels used by the LJTick-InAmp
for ch in [0, 1]:
    ljm.eWriteName(handle, f"AIN{ch}_NEGATIVE_CH", 199)  # Single-ended to GND
    # Range must be 10.0 or 1.0 to accommodate the 1.25V offset baseline safely
    ljm.eWriteName(handle, f"AIN{ch}_RANGE", 10.0) 
    ljm.eWriteName(handle, f"AIN{ch}_RESOLUTION_INDEX", 8)

print("Reading amplified thermocouples using InAmp 3 Transfer Function...\n")

try:
    while True:
        # 1. Read amplified voltages from AIN0 and AIN1
        amp_v1 = ljm.eReadName(handle, "AIN0")
        amp_v2 = ljm.eReadName(handle, "AIN1")

        # 2. Extract true thermocouple voltages using inverse transfer function:
        # Vin = (Vout - 1.25) / Gain
        tc_v1 = (amp_v1 - OFFSET_V) / GAIN
        tc_v2 = (amp_v2 - OFFSET_V) / GAIN

        # 3. Read internal T7 temperature for cold junction compensation
        cj_temp_c = ljm.eReadName(handle, "TEMPERATURE_DEVICE_K") - 273.15

        # 4. Convert to final temperatures
        tc1_c = type_k_temp(tc_v1, cj_temp_c) 
        tc2_c = type_k_temp(tc_v2, cj_temp_c)

        # Output data (displayed in Volts for clear calibration checks)
        print(
            f"TC1: Amp Out={amp_v1:6.4f} V | "
            f"Raw TC={tc_v1*1000:7.4f} mV | "
            f"T={tc1_c:6.2f} °C"
        )

        print(
            f"TC2: Amp Out={amp_v2:6.4f} V | "
            f"Raw TC={tc_v2*1000:7.4f} mV | "
            f"T={tc2_c:6.2f} °C"
        )

        print(f"CJ:  {cj_temp_c:6.2f} °C")
        print("-" * 65)

        time.sleep(1)

except KeyboardInterrupt:
    print("\nStopping data collection.")
    pass

finally:
    ljm.close(handle)

Reading amplified thermocouples using InAmp 3 Transfer Function...

TC1: Amp Out=1.2429 V | Raw TC=-0.1386 mV | T= 24.67 °C
TC2: Amp Out=1.2429 V | Raw TC=-0.1386 mV | T= 24.67 °C
CJ:   28.03 °C
-----------------------------------------------------------------
TC1: Amp Out=1.2429 V | Raw TC=-0.1402 mV | T= 24.64 °C
TC2: Amp Out=1.2429 V | Raw TC=-0.1402 mV | T= 24.64 °C
CJ:   28.04 °C
-----------------------------------------------------------------
TC1: Amp Out=1.2427 V | Raw TC=-0.1433 mV | T= 24.56 °C
TC2: Amp Out=1.2429 V | Raw TC=-0.1402 mV | T= 24.63 °C
CJ:   28.03 °C
-----------------------------------------------------------------
TC1: Amp Out=1.2430 V | Raw TC=-0.1371 mV | T= 24.71 °C
TC2: Amp Out=1.2426 V | Raw TC=-0.1448 mV | T= 24.52 °C
CJ:   28.03 °C
-----------------------------------------------------------------
TC1: Amp Out=1.2429 V | Raw TC=-0.1402 mV | T= 24.64 °C
TC2: Amp Out=1.2427 V | Raw TC=-0.1433 mV | T= 24.57 °C
CJ:   28.04 °C
---------------------------------